In [ ]:
import os
import shutil
import subprocess
import sys

# 1. Ensure working directory is VoiceStudio
if not os.path.exists("frontend"):
    %cd /kaggle/working/VoiceStudio

# 2. Add Bun to current Python PATH environment
os.environ["PATH"] = f"/root/.bun/bin:{os.environ['PATH']}"
os.environ["UV_HTTP_TIMEOUT"] = "300"

# 3. System audio libraries
!apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1

# 4. Install UV package installer
!pip install -q uv

# 5. Build frontend web renderer (Vite bundle)
!bun install --frozen-lockfile
!bun run --cwd electron build:web

# 6. Install Python dependencies with Torch constraints
!uv pip install --system --no-cache --constraint deploy/torch-constraints.txt .

In [ ]:
from huggingface_hub import snapshot_download

model_path = snapshot_download("k2-fsa/OmniVoice")
print("OmniVoice model cached at:", model_path)

In [ ]:
import os
import subprocess
import sys
import time

guard_file = "/kaggle/working/VoiceStudio/backend/core/browser_guard.py"

with open(guard_file, "r") as f:
    guard_code = f.read()

# Force the middleware refusal check to always return None (allow all tunnel hosts & origins)
guard_code = guard_code.replace(
    "detail = self._refusal(scope, connection, method)", "detail = None"
)

with open(guard_file, "w") as f:
    f.write(guard_code)

print("browser_guard.py patched to allow all tunnel hosts.")

# Restart the backend server
PORT = 3900
subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
time.sleep(1)

env = os.environ.copy()
env["OMNIVOICE_SERVER_MODE"] = "1"
env["OMNIVOICE_ALLOWED_HOSTS"] = "*"
env["OMNIVOICE_DATA_DIR"] = "/kaggle/working/omnivoice_data"
env["PYTHONUNBUFFERED"] = "1"

log = open("/kaggle/working/backend.log", "ab")
subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "main:app",
        "--app-dir",
        "backend",
        "--host",
        "0.0.0.0",
        "--port",
        str(PORT),
    ],
    cwd="/kaggle/working/VoiceStudio",
    env=env,
    stdout=log,
    stderr=subprocess.STDOUT,
)

time.sleep(6)
 
# Quick verification test simulating an external host
print("Testing /engines with external Host header:")
!curl -s -o /dev/null -w "HTTP Status: %{http_code}\n" -H "Origin: https://example.com" -H "Host: example.com" http://127.0.0.1:3900/engines

In [ ]:
# Download cloudflared binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Launch tunnel targeting the VoiceStudio port
import re
import subprocess

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:3900"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in tunnel.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print(f"\n==========================================")
        print(f"VOICESTUDIO WEB UI URL: {match.group(0)}")
        print(f"==========================================\n")
        break